# DataSage 2/4 — Enrichment Inference

Loads the **enrichment** GRPO LoRA adapter and runs inference against the live HF Space environment.  
Also runs the **base model** (no LoRA) on the same seeds for comparison.

| Setting | Value |
|---------|-------|
| Runtime | **GPU** (T4 minimum, A100 recommended) |
| Secrets | `HF_TOKEN` (optional, models are public) |
| LoRA adapter | `ricalanis/enrichment-grpo` |
| Base model | `unsloth/Qwen2.5-3B-Instruct` |
| Environment | `https://ricalanis-datasage-enrichment.hf.space` |
| Output | `enrichment_results.json` |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/02_enrichment_inference.ipynb)

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl requests numpy datasets huggingface_hub

In [ ]:
# Cell 2: Imports, Colab detection, GPU check
import os
import sys
import json
import re
import time
import requests
import numpy as np
import torch
from datetime import datetime

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("Loaded HF_TOKEN from Colab Secrets")
    except Exception:
        print("HF_TOKEN not in Secrets (OK — models are public)")
else:
    print("Running locally")

assert torch.cuda.is_available(), (
    "No GPU detected! Go to: Runtime > Change runtime type > GPU (T4 or better)"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: All configuration

BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
LORA_REPO  = "ricalanis/enrichment-grpo"
MAX_SEQ_LENGTH = 2048

ENV_URL = "https://ricalanis-datasage-enrichment.hf.space"

DOMAINS = ["hr", "sales", "pm", "it_ops"]
EPISODES_PER_DOMAIN = 3
BASE_EPISODES_PER_DOMAIN = 1
MAX_STEPS = 12

SYSTEM_PROMPT = (
    "You are a data enrichment agent. You enrich enterprise datasets by adding "
    "computed fields and external lookups.\n\n"
    "Each turn, respond with a JSON enrichment action:\n"
    '{"operation": "<op>", "field_name": "<name>", "source": "<source>", "logic": "", "params": {}}\n\n'
    "Available operations: add_field, lookup, compute_derived, add_category.\n"
    "Pick from the possible_enrichments list. Use a different field each time."
)

# All valid enrichment field names across domains
ENRICHMENT_SOURCES = [
    "salary_band", "tenure_risk", "satisfaction_index", "industry_benchmark",
    "flight_risk_score", "deal_size_category", "velocity_score",
    "win_probability_model", "industry_code", "competitive_risk",
    "schedule_risk_score", "resource_utilization", "dependency_chain_depth",
    "burndown_rate", "delay_probability", "sla_compliance_flag", "mttr_band",
    "escalation_path", "incident_severity_score", "recurring_pattern_flag",
]

print("Config loaded.")
print(f"  LoRA:    {LORA_REPO}")
print(f"  Env:     {ENV_URL}")
print(f"  Domains: {DOMAINS}")
print(f"  Max steps: {MAX_STEPS}")

In [ ]:
# Cell 4: Helper functions

import json
import re
import time
import requests
import numpy as np
import torch


def parse_action(text):
    """Parse model output into an enrichment action dict."""
    m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            if "field_name" in data:
                return data
        except json.JSONDecodeError:
            pass
    tl = text.lower()
    for s in ENRICHMENT_SOURCES:
        if s.replace("_", " ") in tl or s in tl:
            return {"operation": "add_field", "field_name": s, "source": s, "logic": "", "params": {}}
    return {"operation": "add_field", "field_name": "unknown", "source": "", "logic": "", "params": {}}


def env_reset(seed=42):
    try:
        r = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed}, timeout=30)
        r.raise_for_status()
        d = r.json()
        return d.get("observation", d)
    except Exception as e:
        print(f"  [WARN] env_reset failed: {e}")
        return {"error": str(e)}


def env_step(action):
    try:
        r = requests.post(f"{ENV_URL}/web/step", json={"action": action}, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"  [WARN] env_step failed: {e}")
        return {"error": str(e), "reward": 0, "done": True, "observation": {}}


def format_obs(obs):
    return (
        f"Domain: {obs.get('domain', 'unknown')}\n"
        f"Schema: {obs.get('schema_info', '')}\n"
        f"Coverage: {obs.get('enrichment_coverage', 0)}\n"
        f"Fields Added: {obs.get('fields_added', [])}\n"
        f"Possible Enrichments: {obs.get('possible_enrichments', [])}\n"
        f"Step {obs.get('step_number', '?')}/{obs.get('max_steps', '?')}\n\n"
        "Output a single JSON enrichment action. Choose a field NOT already added."
    )


def generate(model, tokenizer, sys_prompt, user_prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    return response


def run_episode(model, tokenizer, seed=42):
    obs = env_reset(seed)
    if "error" in obs:
        return {"error": obs["error"]}
    rewards = []
    actions = []
    for _ in range(MAX_STEPS):
        resp = generate(model, tokenizer, SYSTEM_PROMPT, format_obs(obs))
        action = parse_action(resp)
        actions.append(action)
        result = env_step(action)
        rewards.append(result.get("reward", 0))
        new_obs = result.get("observation", {})
        if isinstance(new_obs, dict):
            obs = {**obs, **new_obs}
        if result.get("done", False):
            break
    coverage = obs.get("enrichment_coverage", 0)
    return {
        "final_coverage": float(coverage),
        "rewards": [float(r) for r in rewards],
        "avg_reward": float(np.mean(rewards)) if rewards else 0.0,
        "steps": len(rewards),
        "fields_added": len([a for a in actions if a.get("field_name", "unknown") != "unknown"]),
        "actions": actions,
    }


try:
    r = requests.get(ENV_URL, timeout=10)
    print(f"Environment OK: {ENV_URL} (status {r.status_code})")
except Exception as e:
    print(f"WARNING: Environment UNREACHABLE: {ENV_URL} ({e})")

print("All helpers defined.")

In [ ]:
# Cell 5: Load LoRA model
from unsloth import FastLanguageModel

print(f"Loading LoRA adapter: {LORA_REPO}")
t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_REPO, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model)
print(f"Loaded in {time.time()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 6: Run LoRA inference
print("=" * 60)
print(f"ENRICHMENT — DataSage LoRA ({LORA_REPO})")
print(f"  {len(DOMAINS)} domains x {EPISODES_PER_DOMAIN} episodes = {len(DOMAINS)*EPISODES_PER_DOMAIN} total")
print("=" * 60)

lora_results = []
for di, domain in enumerate(DOMAINS):
    for ep in range(EPISODES_PER_DOMAIN):
        seed = di * 100 + ep + 1
        print(f"  [{domain}] ep {ep+1}/{EPISODES_PER_DOMAIN} seed={seed}", end=" ... ")
        t0 = time.time()
        result = run_episode(model, tokenizer, seed)
        result["domain"] = domain
        result["seed"] = seed
        result["latency_s"] = round(time.time() - t0, 1)
        lora_results.append(result)
        if "error" in result:
            print(f"ERROR: {result['error']}")
        else:
            print(f"cov={result['final_coverage']:.4f} R={result['avg_reward']:.4f} steps={result['steps']} ({result['latency_s']}s)")

valid = [r for r in lora_results if "error" not in r]
if valid:
    print(f"\n--- LoRA Summary ({len(valid)} episodes) ---")
    print(f"  Mean Coverage: {np.mean([r['final_coverage'] for r in valid]):.4f}")
    print(f"  Mean Reward:   {np.mean([r['avg_reward'] for r in valid]):.4f}")

del model, tokenizer
torch.cuda.empty_cache()
print(f"\nGPU freed. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 7: Load base model
from unsloth import FastLanguageModel

print(f"Loading base model: {BASE_MODEL}")
t0 = time.time()
base_m, base_t = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
if base_t.pad_token is None:
    base_t.pad_token = base_t.eos_token
FastLanguageModel.for_inference(base_m)
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Cell 8: Run base model inference
print(f"\nBASE MODEL — Enrichment ({BASE_MODEL})")

base_results = []
for di, domain in enumerate(DOMAINS):
    for ep in range(BASE_EPISODES_PER_DOMAIN):
        seed = di * 100 + ep + 1
        print(f"  [{domain}] seed={seed}", end=" ... ")
        result = run_episode(base_m, base_t, seed)
        result["domain"] = domain
        result["seed"] = seed
        base_results.append(result)
        if "error" in result:
            print(f"ERROR: {result['error']}")
        else:
            print(f"cov={result['final_coverage']:.4f} R={result['avg_reward']:.4f}")

del base_m, base_t
torch.cuda.empty_cache()
print("Done. GPU freed.")

In [ ]:
# Cell 9: Summarize and save
import os
import sys
import json
import numpy as np
from datetime import datetime

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def summarize(results):
    valid = [r for r in results if "error" not in r]
    if not valid:
        return {"error": "no valid episodes"}
    return {
        "final_coverage_mean": float(np.mean([r["final_coverage"] for r in valid])),
        "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid])),
        "steps_mean": float(np.mean([r["steps"] for r in valid])),
        "fields_added_mean": float(np.mean([r["fields_added"] for r in valid])),
        "n_episodes": len(valid),
        "per_domain": {
            d: {
                "final_coverage_mean": float(np.mean([r["final_coverage"] for r in valid if r.get("domain") == d])),
                "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid if r.get("domain") == d])),
                "n": len([r for r in valid if r.get("domain") == d]),
            }
            for d in DOMAINS
            if any(r.get("domain") == d for r in valid)
        },
    }


output = {
    "task": "enrichment",
    "datasage_lora": summarize(lora_results),
    "base_model": summarize(base_results),
    "raw_lora": [{k: v for k, v in r.items() if k != "actions"} for r in lora_results],
    "raw_base": [{k: v for k, v in r.items() if k != "actions"} for r in base_results],
    "metadata": {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "lora_repo": LORA_REPO, "base_model": BASE_MODEL,
        "env_url": ENV_URL, "domains": DOMAINS,
        "episodes_per_domain": EPISODES_PER_DOMAIN,
        "base_episodes_per_domain": BASE_EPISODES_PER_DOMAIN,
        "max_steps": MAX_STEPS,
    },
}

OUTPUT_FILE = "enrichment_results.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved: {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE)/1024:.1f} KB)")

ls = output["datasage_lora"]
bs = output["base_model"]
if "error" not in ls and "error" not in bs:
    print(f"\n  LoRA  cov={ls['final_coverage_mean']:.4f}  R={ls['avg_reward_mean']:.4f}  ({ls['n_episodes']} eps)")
    print(f"  Base  cov={bs['final_coverage_mean']:.4f}  R={bs['avg_reward_mean']:.4f}  ({bs['n_episodes']} eps)")
    print(f"  Delta cov={ls['final_coverage_mean']-bs['final_coverage_mean']:+.4f}")

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print(f"\nDownloaded {OUTPUT_FILE} to your browser.")